In [1]:
import os
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim

from torchvision import datasets, transforms, models
from torch.utils.data import DataLoader, WeightedRandomSampler

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [2]:
import torch
import gc

torch.cuda.empty_cache()
gc.collect()

190

In [3]:
train_transform = transforms.Compose([
    transforms.Resize((224,224)),

    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomRotation(15),
    transforms.ColorJitter(brightness=0.2, contrast=0.2),

    transforms.ToTensor(),

    transforms.Normalize(
        [0.485, 0.456, 0.406],
        [0.229, 0.224, 0.225]
    )
])

val_transform = transforms.Compose([
    transforms.Resize((224,224)),
    transforms.ToTensor(),

    transforms.Normalize(
        [0.485, 0.456, 0.406],
        [0.229, 0.224, 0.225]
    )
])

In [4]:
DATA_DIR = "/kaggle/input/datasets/venkatsaikondra/vnkt12/Final_Data"

train_dataset = datasets.ImageFolder(os.path.join(DATA_DIR, "train"), transform=train_transform)
val_dataset   = datasets.ImageFolder(os.path.join(DATA_DIR, "val"), transform=val_transform)
test_dataset  = datasets.ImageFolder(os.path.join(DATA_DIR, "test"), transform=val_transform)

class_names = train_dataset.classes
print("Classes:", class_names)

Classes: ['Covid-19', 'Normal', 'Pneumonia-Bacterial', 'Pneumonia-Viral']


In [5]:
targets = [label for _, label in train_dataset]
class_counts = np.bincount(targets)

class_weights = 1.0 / class_counts
sample_weights = [class_weights[t] for t in targets]

sampler = WeightedRandomSampler(sample_weights, len(sample_weights), replacement=True)

In [6]:
train_loader = DataLoader(train_dataset, batch_size=8, sampler=sampler)
val_loader   = DataLoader(val_dataset, batch_size=8, shuffle=False)
test_loader  = DataLoader(test_dataset, batch_size=8, shuffle=False)

In [7]:
effnet = models.efficientnet_b4(weights="IMAGENET1K_V1")
effnet.classifier[1] = nn.Linear(effnet.classifier[1].in_features, 4)

from torchvision.models import inception_v3, Inception_V3_Weights

inception = inception_v3(weights=Inception_V3_Weights.IMAGENET1K_V1)

# Main classifier
inception.fc = nn.Linear(inception.fc.in_features, 4)

# 🔥 AUX CLASSIFIER FIX (THIS IS YOUR ERROR SOURCE)
inception.AuxLogits.fc = nn.Linear(inception.AuxLogits.fc.in_features, 4)

inception = inception.to(device)

effnet = effnet.to(device)

In [8]:
class_weights_tensor = torch.tensor([1.0, 1.0, 1.1, 1.1]).to(device)

criterion = nn.CrossEntropyLoss(weight=class_weights_tensor, label_smoothing=0.1)

optimizer_eff = optim.AdamW(effnet.parameters(), lr=3e-4)
optimizer_inc = optim.AdamW(inception.parameters(), lr=3e-4)

In [9]:
import torch.nn.functional as F

def resize_for_models(images):
    eff_imgs = F.interpolate(images, size=(224,224), mode='bilinear', align_corners=False)
    inc_imgs = F.interpolate(images, size=(299,299), mode='bilinear', align_corners=False)
    return eff_imgs, inc_imgs

In [10]:
def train_one_epoch_dual(effnet, inception, loader, opt_eff, opt_inc):

    effnet.train()
    inception.train()

    total_loss = 0

    for images, labels in loader:

        images = images.to(device)
        labels = labels.to(device)

        # 🔥 CRITICAL FIX
        eff_imgs, inc_imgs = resize_for_models(images)

        # ---- EfficientNet ----
        opt_eff.zero_grad()
        out_eff = effnet(eff_imgs)
        loss_eff = criterion(out_eff, labels)
        loss_eff.backward()
        opt_eff.step()

        # ---- Inception ----
        opt_inc.zero_grad()
        out_inc = inception(inc_imgs)

        if isinstance(out_inc, tuple):
            main, aux = out_inc
            loss_inc = criterion(main, labels) + 0.4 * criterion(aux, labels)
        else:
            loss_inc = criterion(out_inc, labels)

        loss_inc.backward()
        opt_inc.step()

        total_loss += (loss_eff.item() + loss_inc.item())

    return total_loss / len(loader)

In [11]:
EPOCHS = 30

for epoch in range(EPOCHS):

    loss = train_one_epoch_dual(
        effnet,
        inception,
        train_loader,
        optimizer_eff,
        optimizer_inc
    )

    print(f"Epoch {epoch+1}: Loss = {loss:.4f}")

Epoch 1: Loss = 1.8713
Epoch 2: Loss = 1.6183
Epoch 3: Loss = 1.5233
Epoch 4: Loss = 1.4715
Epoch 5: Loss = 1.3957
Epoch 6: Loss = 1.3688
Epoch 7: Loss = 1.3324
Epoch 8: Loss = 1.2913
Epoch 9: Loss = 1.2399
Epoch 10: Loss = 1.2239
Epoch 11: Loss = 1.1982
Epoch 12: Loss = 1.1755
Epoch 13: Loss = 1.1541
Epoch 14: Loss = 1.1361
Epoch 15: Loss = 1.1112
Epoch 16: Loss = 1.1082
Epoch 17: Loss = 1.0866
Epoch 18: Loss = 1.0882
Epoch 19: Loss = 1.0557
Epoch 20: Loss = 1.0510
Epoch 21: Loss = 1.0462
Epoch 22: Loss = 1.0132
Epoch 23: Loss = 1.0147
Epoch 24: Loss = 0.9982
Epoch 25: Loss = 1.0092
Epoch 26: Loss = 0.9976
Epoch 27: Loss = 0.9822
Epoch 28: Loss = 0.9915
Epoch 29: Loss = 0.9789
Epoch 30: Loss = 0.9836


In [12]:
def safe_forward(model, x):
    out = model(x)
    if isinstance(out, tuple):   # Inception fix
        out = out[0]
    return out

In [13]:
effnet.eval()
inception.eval()

Inception3(
  (Conv2d_1a_3x3): BasicConv2d(
    (conv): Conv2d(3, 32, kernel_size=(3, 3), stride=(2, 2), bias=False)
    (bn): BatchNorm2d(32, eps=0.001, momentum=0.1, affine=True, track_running_stats=True)
  )
  (Conv2d_2a_3x3): BasicConv2d(
    (conv): Conv2d(32, 32, kernel_size=(3, 3), stride=(1, 1), bias=False)
    (bn): BatchNorm2d(32, eps=0.001, momentum=0.1, affine=True, track_running_stats=True)
  )
  (Conv2d_2b_3x3): BasicConv2d(
    (conv): Conv2d(32, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
    (bn): BatchNorm2d(64, eps=0.001, momentum=0.1, affine=True, track_running_stats=True)
  )
  (maxpool1): MaxPool2d(kernel_size=3, stride=2, padding=0, dilation=1, ceil_mode=False)
  (Conv2d_3b_1x1): BasicConv2d(
    (conv): Conv2d(64, 80, kernel_size=(1, 1), stride=(1, 1), bias=False)
    (bn): BatchNorm2d(80, eps=0.001, momentum=0.1, affine=True, track_running_stats=True)
  )
  (Conv2d_4a_3x3): BasicConv2d(
    (conv): Conv2d(80, 192, kernel_size=(3, 3), stri

In [20]:
import torch
import torch.nn.functional as F
from sklearn.metrics import f1_score, accuracy_score
import pandas as pd

# ✅ Correct model names
effnet.eval()
inception.eval()

boost_values = [1.0, 1.1, 1.2, 1.3, 1.4, 1.5, 1.6, 1.7]
results = []

# 🔥 Safe forward (handles Inception tuple output)
def safe_forward(model, x):
    out = model(x)
    if isinstance(out, tuple):
        out = out[0]
    return out


def ensemble_predict(img224, img299, boost):

    logits1 = safe_forward(effnet, img224)
    logits2 = safe_forward(inception, img299)

    logits = 0.5 * logits1 + 0.5 * logits2
    logits = logits * boost

    return torch.softmax(logits, dim=1)


# ✅ CLINICALLY VALID TTA
def tta_predict(images, boost):

    outputs = []

    img224 = images
    img299 = F.interpolate(images, size=(299, 299), mode='bilinear', align_corners=False)

    transforms = [
        lambda x: x,
        lambda x: torch.flip(x, dims=[3]),  # horizontal flip only
    ]

    for t in transforms:
        aug224 = t(img224)
        aug299 = t(img299)

        out = ensemble_predict(aug224, aug299, boost)
        outputs.append(out)

    return torch.mean(torch.stack(outputs), dim=0)


# 🔁 Ablation loop
for b in boost_values:

    print(f"\n🔥 Testing boost = {b}")

    boost = torch.tensor([1.0, 1.0, b, b], device=device)

    all_preds, all_labels = [], []

    with torch.no_grad():
        for images, labels in test_loader:

            images = images.to(device)
            labels = labels.to(device)

            outputs = tta_predict(images, boost)
            preds = torch.argmax(outputs, dim=1)

            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())

    f1 = f1_score(all_labels, all_preds, average='macro')
    acc = accuracy_score(all_labels, all_preds)

    print(f"F1: {f1:.4f} | Acc: {acc:.4f}")

    results.append({
        "Boost (Bacterial/Viral)": b,
        "F1 Score": round(f1, 4),
        "Accuracy": round(acc, 4)
    })

df_boost = pd.DataFrame(results)
print("\n📊 Boost Ablation Results:")
print(df_boost)


🔥 Testing boost = 1.0
F1: 0.9457 | Acc: 0.9457

🔥 Testing boost = 1.1
F1: 0.9457 | Acc: 0.9457

🔥 Testing boost = 1.2
F1: 0.9463 | Acc: 0.9463

🔥 Testing boost = 1.3
F1: 0.9463 | Acc: 0.9463

🔥 Testing boost = 1.4
F1: 0.9463 | Acc: 0.9463

🔥 Testing boost = 1.5
F1: 0.9463 | Acc: 0.9463

🔥 Testing boost = 1.6
F1: 0.9463 | Acc: 0.9463

🔥 Testing boost = 1.7
F1: 0.9463 | Acc: 0.9463

📊 Boost Ablation Results:
   Boost (Bacterial/Viral)  F1 Score  Accuracy
0                      1.0    0.9457    0.9457
1                      1.1    0.9457    0.9457
2                      1.2    0.9463    0.9463
3                      1.3    0.9463    0.9463
4                      1.4    0.9463    0.9463
5                      1.5    0.9463    0.9463
6                      1.6    0.9463    0.9463
7                      1.7    0.9463    0.9463


In [23]:
weight_pairs = [
    (1.0, 0.0),
    (0.9, 0.1),
    (0.7, 0.3),
    (0.5, 0.5),
    (0.3, 0.7),
    (0.1, 0.9),
    (0.0, 1.0)
]

boost = torch.tensor([1.0, 1.0, 1.2, 1.2], device=device)
results = []


def ensemble_predict(img224, img299, w1, w2, boost):

    logits1 = safe_forward(effnet, img224)
    logits2 = safe_forward(inception, img299)

    logits = w1 * logits1 + w2 * logits2
    logits = logits * boost

    return torch.softmax(logits, dim=1)


def tta_predict(images, w1, w2, boost):

    outputs = []

    img224 = images
    img299 = F.interpolate(images, size=(299, 299), mode='bilinear', align_corners=False)

    transforms = [
        lambda x: x,
        lambda x: torch.flip(x, dims=[3]),
    ]

    for t in transforms:
        aug224 = t(img224)
        aug299 = t(img299)

        out = ensemble_predict(aug224, aug299, w1, w2, boost)
        outputs.append(out)

    return torch.mean(torch.stack(outputs), dim=0)


# 🔁 Ablation loop
for w1, w2 in weight_pairs:

    print(f"\n🔥 Testing weights = ({w1}, {w2})")

    all_preds, all_labels = [], []

    with torch.no_grad():
        for images, labels in test_loader:

            images = images.to(device)
            labels = labels.to(device)

            outputs = tta_predict(images, w1, w2, boost)
            preds = torch.argmax(outputs, dim=1)

            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())

    f1 = f1_score(all_labels, all_preds, average='macro')
    acc = accuracy_score(all_labels, all_preds)

    print(f"F1: {f1:.4f} | Acc: {acc:.4f}")

    results.append({
        "Weights (EffNet/Inception)": f"{w1}/{w2}",
        "F1 Score": round(f1, 4),
        "Accuracy": round(acc, 4)
    })

df_weights = pd.DataFrame(results)
print("\n📊 Weight Ablation Results:")
print(df_weights)


🔥 Testing weights = (1.0, 0.0)
F1: 0.9421 | Acc: 0.9420

🔥 Testing weights = (0.9, 0.1)
F1: 0.9439 | Acc: 0.9438

🔥 Testing weights = (0.7, 0.3)
F1: 0.9482 | Acc: 0.9481

🔥 Testing weights = (0.5, 0.5)
F1: 0.9463 | Acc: 0.9463

🔥 Testing weights = (0.3, 0.7)
F1: 0.9395 | Acc: 0.9395

🔥 Testing weights = (0.1, 0.9)
F1: 0.9234 | Acc: 0.9235

🔥 Testing weights = (0.0, 1.0)
F1: 0.9201 | Acc: 0.9204

📊 Weight Ablation Results:
  Weights (EffNet/Inception)  F1 Score  Accuracy
0                    1.0/0.0    0.9421    0.9420
1                    0.9/0.1    0.9439    0.9438
2                    0.7/0.3    0.9482    0.9481
3                    0.5/0.5    0.9463    0.9463
4                    0.3/0.7    0.9395    0.9395
5                    0.1/0.9    0.9234    0.9235
6                    0.0/1.0    0.9201    0.9204


In [24]:
def msw_ensnet_predict(images):

    images = images.to(device)

    eff_imgs, inc_imgs = resize_for_models(images)

    out_eff = torch.softmax(safe_forward(effnet, eff_imgs), dim=1)
    out_inc = torch.softmax(safe_forward(inception, inc_imgs), dim=1)

    fused = 0.7 * out_eff + 0.3 * out_inc

    boost = torch.tensor([1.0,1.0,1.2,1.2]).to(device)
    fused = fused * boost

    fused = fused / fused.sum(dim=1, keepdim=True)

    return fused

In [25]:
import random
import numpy as np
import torch

seed = 42
random.seed(seed)
np.random.seed(seed)
torch.manual_seed(seed)
torch.cuda.manual_seed_all(seed)

In [26]:
test_loader = DataLoader(test_dataset, batch_size=4, shuffle=False)

In [ ]:
print(len(test_dataset))

In [27]:
from sklearn.metrics import classification_report

all_preds = []
all_labels = []

effnet.eval()
inception.eval()

with torch.no_grad():

    for images, labels in test_loader:

        images = images.to(device)
        labels = labels.to(device)

        outputs = tta_predict(images, 0.7, 0.3, boost)  # or your final config
        preds = torch.argmax(outputs, dim=1)

        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())

print(classification_report(all_labels, all_preds, target_names=class_names))

                     precision    recall  f1-score   support

           Covid-19       1.00      1.00      1.00       405
             Normal       0.99      0.98      0.98       405
Pneumonia-Bacterial       0.93      0.89      0.91       405
    Pneumonia-Viral       0.88      0.93      0.90       405

           accuracy                           0.95      1620
          macro avg       0.95      0.95      0.95      1620
       weighted avg       0.95      0.95      0.95      1620



In [28]:
def tta_predict(image):
    transforms_list = [
        lambda x: x,
        lambda x: transforms.functional.hflip(x),
        lambda x: transforms.functional.rotate(x, 5),
        lambda x: transforms.functional.rotate(x, -5),
    ]

    preds = []

    for t in transforms_list:
        aug = t(image)
        aug = aug.unsqueeze(0)

        pred = msw_ensnet_predict(aug)
        preds.append(pred)

    return torch.mean(torch.stack(preds), dim=0)

In [48]:
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix, cohen_kappa_score, roc_auc_score
)
import numpy as np
import torch

import numpy as np
import torch

def evaluate_model(model_fn, loader, use_tta=False):

    all_preds, all_probs, all_labels = [], [], []

    if hasattr(model_fn, "eval"):
        model_fn.eval()

    with torch.no_grad():

        for images, labels in loader:

            images = images.to(device)

            # ✅ FIXED PARAM NAME
            if use_tta:
                outputs = tta_batch(images, model_fn)
            else:
                outputs = model_fn(images)

            # 🔥 Handle Inception tuple
            if isinstance(outputs, tuple):
                outputs = outputs[0]

            probs = torch.softmax(outputs, dim=1).detach().cpu().numpy()
            preds = np.argmax(probs, axis=1)

            all_preds.extend(preds)
            all_probs.extend(probs)
            all_labels.extend(labels.numpy())

    return np.array(all_labels), np.array(all_preds), np.array(all_probs)

In [30]:
import torchvision.transforms.functional as TF

def tta_batch(images, model_fn):
    aug_list = [
        lambda x: x,
        lambda x: TF.hflip(x),
        lambda x: TF.rotate(x, 5),
        lambda x: TF.rotate(x, -5),
        lambda x: TF.adjust_brightness(x, 1.1),
        lambda x: TF.adjust_contrast(x, 1.1),
    ]

    preds = []

    for aug in aug_list:
        aug_imgs = torch.stack([aug(img) for img in images])
        outputs = model_fn(aug_imgs.to(device)).cpu()
        preds.append(torch.softmax(outputs, dim=1))

    return torch.mean(torch.stack(preds), dim=0)

In [31]:
def compute_metrics(y_true, y_pred, y_prob):

    acc = accuracy_score(y_true, y_pred)

    precision = precision_score(y_true, y_pred, average='macro')
    recall = recall_score(y_true, y_pred, average='macro')
    f1 = f1_score(y_true, y_pred, average='macro')

    bal_acc = np.mean(recall_score(y_true, y_pred, average=None))

    kappa = cohen_kappa_score(y_true, y_pred)

    # One-vs-Rest AUC
    auc = roc_auc_score(y_true, y_prob, multi_class='ovr')

    # Confusion matrix → specificity
    cm = confusion_matrix(y_true, y_pred)
    specificity = []

    for i in range(len(cm)):
        tn = cm.sum() - (cm[i,:].sum() + cm[:,i].sum() - cm[i,i])
        fp = cm[:,i].sum() - cm[i,i]
        specificity.append(tn / (tn + fp))

    specificity = np.mean(specificity)

    return {
        "accuracy": acc,
        "precision": precision,
        "recall": recall,
        "f1": f1,
        "balanced_accuracy": bal_acc,
        "specificity": specificity,
        "auc": auc,
        "kappa": kappa
    }

In [32]:
def bootstrap_ci(y_true, y_pred, n=1000):
    scores = []
    N = len(y_true)

    for _ in range(n):
        idx = np.random.choice(N, N, replace=True)
        score = f1_score(y_true[idx], y_pred[idx], average='macro')
        scores.append(score)

    lower = np.percentile(scores, 2.5)
    upper = np.percentile(scores, 97.5)

    return lower, upper

In [33]:
from statsmodels.stats.contingency_tables import mcnemar

def mcnemar_test(y_true, pred1, pred2):

    b = np.sum((pred1 == y_true) & (pred2 != y_true))
    c = np.sum((pred1 != y_true) & (pred2 == y_true))

    table = [[0, b], [c, 0]]

    result = mcnemar(table, exact=True)
    return result.pvalue

In [36]:
def safe_forward(model, x):
    out = model(x)
    if isinstance(out, tuple):
        out = out[0]
    return out

In [49]:
results = {}

# 🔒 Set eval mode (IMPORTANT)
effnet.eval()
inception.eval()


# 🔥 Safe forward (handles Inception tuple output)
def safe_forward(model, x):
    out = model(x)
    if isinstance(out, tuple):
        out = out[0]
    return out


# ==============================
# ✅ EfficientNet
# ==============================
y_true, y_pred, y_prob = evaluate_model(
    lambda x: safe_forward(effnet, x),
    test_loader,
    use_tta=False
)
results["EffNet"] = compute_metrics(y_true, y_pred, y_prob)


# ==============================
# ✅ Inception (with 299 resize)
# ==============================
def inception_fn(x):
    x299 = torch.nn.functional.interpolate(x, size=(299, 299), mode='bilinear', align_corners=False)
    return safe_forward(inception, x299)

y_true, y_pred, y_prob = evaluate_model(
    inception_fn,
    test_loader,
    use_tta=False
)
results["Inception"] = compute_metrics(y_true, y_pred, y_prob)


# ==============================
# ✅ MSW-EnsNet WITHOUT TTA
# ==============================
def ensemble_no_tta(x):

    x224 = x
    x299 = torch.nn.functional.interpolate(x, size=(299, 299), mode='bilinear', align_corners=False)

    out_eff = torch.softmax(safe_forward(effnet, x224), dim=1)
    out_inc = torch.softmax(safe_forward(inception, x299), dim=1)

    fused = 0.7 * out_eff + 0.3 * out_inc

    return fused


y_true, y_pred, y_prob = evaluate_model(
    ensemble_no_tta,
    test_loader,
    use_tta=False
)
results["MSW_no_TTA"] = compute_metrics(y_true, y_pred, y_prob)


# ==============================
# ✅ MSW-EnsNet WITH TTA
# ==============================
def ensemble_fn(x):
    return msw_ensnet_predict(x)  # already includes dual input + boost


y_true, y_pred, y_prob = evaluate_model(
    ensemble_fn,
    test_loader,
    use_tta=True
)
results["MSW_TTA"] = compute_metrics(y_true, y_pred, y_prob)


# ==============================
# 📊 PRINT RESULTS
# ==============================
for name, metrics in results.items():
    print(f"\n{name}")
    for k, v in metrics.items():
        print(f"{k}: {v:.4f}")


EffNet
Accuracy: 0.9414
Precision: 0.9426
Recall: 0.9414
F1: 0.9414
Balanced_Accuracy: 0.9414
Specificity: 0.9805
AUC: 0.9885
Kappa: 0.9218

Inception
Accuracy: 0.9086
Precision: 0.9083
Recall: 0.9086
F1: 0.9084
Balanced_Accuracy: 0.9086
Specificity: 0.9695
AUC: 0.9856
Kappa: 0.8782

MSW_no_TTA
Accuracy: 0.9420
Precision: 0.9433
Recall: 0.9420
F1: 0.9420
Balanced_Accuracy: 0.9420
Specificity: 0.9807
AUC: 0.9928
Kappa: 0.9226

MSW_TTA
Accuracy: 0.9463
Precision: 0.9483
Recall: 0.9463
F1: 0.9465
Balanced_Accuracy: 0.9463
Specificity: 0.9821
AUC: 0.9903
Kappa: 0.9284


In [50]:
def no_boost(x):
    out_eff = torch.softmax(effnet(x), dim=1)
    out_inc = torch.softmax(inception(x), dim=1)
    return 0.7*out_eff + 0.3*out_inc

In [51]:
import numpy as np
import torch
import torch.nn as nn
import torchvision.transforms.functional as TF

from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix, cohen_kappa_score, roc_auc_score
)

from statsmodels.stats.contingency_tables import mcnemar

In [52]:
def tta_batch(images, model_fn):
    aug_list = [
        lambda x: x,
        lambda x: TF.hflip(x),
        lambda x: TF.rotate(x, 5),
        lambda x: TF.rotate(x, -5),
        lambda x: TF.adjust_brightness(x, 1.1),
        lambda x: TF.adjust_contrast(x, 1.1),
    ]

    preds = []

    for aug in aug_list:
        aug_imgs = torch.stack([aug(img) for img in images])
        outputs = model_fn(aug_imgs.to(device)).cpu()
        preds.append(torch.softmax(outputs, dim=1))

    return torch.mean(torch.stack(preds), dim=0)

In [59]:
import numpy as np
import torch

def evaluate_model(model_fn, loader, use_tta=False):

    all_preds, all_probs, all_labels = [], [], []

    with torch.no_grad():   # 🔥 prevents grad tracking

        for images, labels in loader:

            images = images.to(device)

            if use_tta:
                outputs = tta_batch(images, model_fn)
            else:
                outputs = model_fn(images)

            # 🔥 Handle Inception tuple output
            if isinstance(outputs, tuple):
                outputs = outputs[0]

            # ✅ CRITICAL FIX
            probs = torch.softmax(outputs, dim=1).detach().cpu().numpy()
            preds = np.argmax(probs, axis=1)

            all_preds.extend(preds)
            all_probs.extend(probs)
            all_labels.extend(labels.numpy())

    return np.array(all_labels), np.array(all_preds), np.array(all_probs)

In [54]:
def compute_metrics(y_true, y_pred, y_prob):

    acc = accuracy_score(y_true, y_pred)

    precision = precision_score(y_true, y_pred, average='macro')
    recall = recall_score(y_true, y_pred, average='macro')
    f1 = f1_score(y_true, y_pred, average='macro')

    balanced_acc = np.mean(recall_score(y_true, y_pred, average=None))

    kappa = cohen_kappa_score(y_true, y_pred)

    auc = roc_auc_score(y_true, y_prob, multi_class='ovr')

    cm = confusion_matrix(y_true, y_pred)

    specificity = []
    for i in range(len(cm)):
        tn = cm.sum() - (cm[i,:].sum() + cm[:,i].sum() - cm[i,i])
        fp = cm[:,i].sum() - cm[i,i]
        specificity.append(tn / (tn + fp))

    specificity = np.mean(specificity)

    return {
        "Accuracy": acc,
        "Precision": precision,
        "Recall": recall,
        "F1": f1,
        "Balanced_Accuracy": balanced_acc,
        "Specificity": specificity,
        "AUC": auc,
        "Kappa": kappa
    }

In [55]:
def bootstrap_ci(y_true, y_pred, n=1000):
    scores = []
    N = len(y_true)

    for _ in range(n):
        idx = np.random.choice(N, N, replace=True)
        score = f1_score(y_true[idx], y_pred[idx], average='macro')
        scores.append(score)

    return np.percentile(scores, 2.5), np.percentile(scores, 97.5)

In [56]:
def mcnemar_test(y_true, pred1, pred2):
    b = np.sum((pred1 == y_true) & (pred2 != y_true))
    c = np.sum((pred1 != y_true) & (pred2 == y_true))

    table = [[0, b], [c, 0]]
    result = mcnemar(table, exact=True)

    return result.pvalue

In [57]:
def effnet_fn(x):
    return effnet(x)

def inception_fn(x):
    return inception(x)

def ensemble_no_boost(x):
    out1 = torch.softmax(effnet(x), dim=1)
    out2 = torch.softmax(inception(x), dim=1)
    return 0.7*out1 + 0.3*out2

def ensemble_with_boost(x):
    out = ensemble_no_boost(x)
    boost = torch.tensor([1.0,1.0,1.1,1.1]).to(device)
    out = out * boost
    return out / out.sum(dim=1, keepdim=True)

In [60]:
results = {}

effnet.eval()
inception.eval()

# 🔥 Safe forward
def safe_forward(model, x):
    out = model(x)
    if isinstance(out, tuple):
        out = out[0]
    return out


# ==============================
# ✅ EfficientNet
# ==============================
def effnet_fn(x):
    return safe_forward(effnet, x)

y_true, y_pred, y_prob = evaluate_model(effnet_fn, test_loader, use_tta=False)
results["EffNet"] = compute_metrics(y_true, y_pred, y_prob)


# ==============================
# ✅ Inception
# ==============================
def inception_fn(x):
    x299 = torch.nn.functional.interpolate(x, size=(299, 299), mode='bilinear', align_corners=False)
    return safe_forward(inception, x299)

y_true, y_pred, y_prob = evaluate_model(inception_fn, test_loader, use_tta=False)
results["Inception"] = compute_metrics(y_true, y_pred, y_prob)


# ==============================
# ✅ Ensemble (NO BOOST)
# ==============================
def ensemble_no_boost(x):

    x224 = x
    x299 = torch.nn.functional.interpolate(x, size=(299, 299), mode='bilinear', align_corners=False)

    out_eff = torch.softmax(safe_forward(effnet, x224), dim=1)
    out_inc = torch.softmax(safe_forward(inception, x299), dim=1)

    return 0.7 * out_eff + 0.3 * out_inc


y_true, y_pred, y_prob = evaluate_model(ensemble_no_boost, test_loader, use_tta=False)
results["Ensemble_NoBoost"] = compute_metrics(y_true, y_pred, y_prob)


# ==============================
# ✅ Ensemble (BOOST, NO TTA)
# ==============================
def ensemble_with_boost(x):

    x224 = x
    x299 = torch.nn.functional.interpolate(x, size=(299, 299), mode='bilinear', align_corners=False)

    out_eff = torch.softmax(safe_forward(effnet, x224), dim=1)
    out_inc = torch.softmax(safe_forward(inception, x299), dim=1)

    fused = 0.7 * out_eff + 0.3 * out_inc

    boost = torch.tensor([1.0, 1.0, 1.1, 1.1], device=device)
    fused = fused * boost
    fused = fused / fused.sum(dim=1, keepdim=True)

    return fused


y_true_nb, y_pred_nb, y_prob_nb = evaluate_model(ensemble_with_boost, test_loader, use_tta=False)
results["MSW_NoTTA"] = compute_metrics(y_true_nb, y_pred_nb, y_prob_nb)


# ==============================
# ✅ Ensemble + TTA
# ==============================
y_true_tta, y_pred_tta, y_prob_tta = evaluate_model(
    ensemble_with_boost,
    test_loader,
    use_tta=True
)

results["MSW_TTA"] = compute_metrics(y_true_tta, y_pred_tta, y_prob_tta)


# ==============================
# 📊 PRINT RESULTS
# ==============================
for model, res in results.items():
    print(f"\n{model}")
    for k, v in res.items():
        print(f"{k}: {v:.4f}")


EffNet
Accuracy: 0.9414
Precision: 0.9426
Recall: 0.9414
F1: 0.9414
Balanced_Accuracy: 0.9414
Specificity: 0.9805
AUC: 0.9885
Kappa: 0.9218

Inception
Accuracy: 0.9086
Precision: 0.9083
Recall: 0.9086
F1: 0.9084
Balanced_Accuracy: 0.9086
Specificity: 0.9695
AUC: 0.9856
Kappa: 0.8782

Ensemble_NoBoost
Accuracy: 0.9420
Precision: 0.9433
Recall: 0.9420
F1: 0.9420
Balanced_Accuracy: 0.9420
Specificity: 0.9807
AUC: 0.9928
Kappa: 0.9226

MSW_NoTTA
Accuracy: 0.9420
Precision: 0.9433
Recall: 0.9420
F1: 0.9420
Balanced_Accuracy: 0.9420
Specificity: 0.9807
AUC: 0.9925
Kappa: 0.9226

MSW_TTA
Accuracy: 0.9463
Precision: 0.9481
Recall: 0.9463
F1: 0.9464
Balanced_Accuracy: 0.9463
Specificity: 0.9821
AUC: 0.9905
Kappa: 0.9284


In [61]:
for model, res in results.items():
    print(f"\n{model}")
    for k,v in res.items():
        print(f"{k}: {v:.4f}")


EffNet
Accuracy: 0.9414
Precision: 0.9426
Recall: 0.9414
F1: 0.9414
Balanced_Accuracy: 0.9414
Specificity: 0.9805
AUC: 0.9885
Kappa: 0.9218

Inception
Accuracy: 0.9086
Precision: 0.9083
Recall: 0.9086
F1: 0.9084
Balanced_Accuracy: 0.9086
Specificity: 0.9695
AUC: 0.9856
Kappa: 0.8782

Ensemble_NoBoost
Accuracy: 0.9420
Precision: 0.9433
Recall: 0.9420
F1: 0.9420
Balanced_Accuracy: 0.9420
Specificity: 0.9807
AUC: 0.9928
Kappa: 0.9226

MSW_NoTTA
Accuracy: 0.9420
Precision: 0.9433
Recall: 0.9420
F1: 0.9420
Balanced_Accuracy: 0.9420
Specificity: 0.9807
AUC: 0.9925
Kappa: 0.9226

MSW_TTA
Accuracy: 0.9463
Precision: 0.9481
Recall: 0.9463
F1: 0.9464
Balanced_Accuracy: 0.9463
Specificity: 0.9821
AUC: 0.9905
Kappa: 0.9284


In [62]:
ci_low, ci_high = bootstrap_ci(y_true_tta, y_pred_tta)
print("95% CI (F1):", ci_low, ci_high)

95% CI (F1): 0.9359081964272543 0.9564507290481511


In [63]:
p_eff = mcnemar_test(y_true_tta, y_pred_tta, y_pred)
p_inc = mcnemar_test(y_true_tta, y_pred_tta, y_pred_nb)

print("MSW vs EffNet p-value:", p_eff)
print("MSW vs Inception p-value:", p_inc)

MSW vs EffNet p-value: 0.2295229434967041
MSW vs Inception p-value: 0.2295229434967041
